# Net worth over time

Reads the latest run file under `data/shared/state/runs/` and plots month-end net worth (nominal) plus an inflation-adjusted (CPI) series.

The run file is opened read-only via `RunConnection`; all reads go through `rc.items(...)`, which returns tidy DataFrames suitable for pandas groupby / pivot. See ADR-002 for the underlying schema.


In [ ]:
from pathlib import Path

import pandas as pd

from analysis.sim.connection import RunConnection

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = REPO / "data" / "shared" / "state" / "runs"

run_files = sorted(RUNS_DIR.glob("*.sqlite3"))
if not run_files:
    raise FileNotFoundError(f"no run files under {RUNS_DIR}")
RUN_FILE = run_files[-1]
print(f"latest run: {RUN_FILE.name}")

rc = RunConnection(RUN_FILE)


In [ ]:
_META_COLS = [
    "scenario_name", "status", "started_at", "finished_at",
    "code_version", "code_dirty", "data_version", "data_dirty",
    "data_mode", "run_by",
]
run_meta = rc.items("simulation_run")[_META_COLS].copy()
run_meta["code_version"] = run_meta["code_version"].str[:12]
run_meta["data_version"] = run_meta["data_version"].str[:12]
run_meta.T


In [ ]:
nw = rc.items("net_worth").rename(columns={"net_worth": "nominal_cents"})
cpi = rc.items("net_worth_constant_cpi")[["month", "constant_cents"]]
nw = nw.merge(cpi, on="month", how="left")

nw["month"] = pd.to_datetime(nw["month"])
nw["nominal"] = nw["nominal_cents"] / 100
nw["cpi_constant"] = nw["constant_cents"].astype("Float64") / 100
nw = nw.set_index("month")
nw[["nominal", "cpi_constant"]].head()


## Year-end net worth (table)


In [ ]:
year_end = (
    nw.groupby(nw.index.year)
      .tail(1)
      .assign(year=lambda d: d.index.year)
      .set_index("year")[["nominal", "cpi_constant"]]
)
year_end.style.format({"nominal": "${:,.0f}", "cpi_constant": "${:,.0f}"})


## Monthly net worth (chart)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nw.index, nw["nominal"], label="nominal")
if nw["cpi_constant"].notna().any():
    ax.plot(nw.index, nw["cpi_constant"], label="CPI-constant", linestyle="--")
ax.set_title(
    f"Net worth — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, alpha=0.3)
ax.legend()
fig.autofmt_xdate()
plt.show()


## Income by category

Income flows are transfers _out of_ accounts with `kind = 'income'`. Subkinds are bucketed:

- **salary** — `salary`
- **investment** — `interest`, `dividend`, `capital_gain`
- **other** — `ss_benefit`, `pension`, `rental`, `other_income`


In [ ]:
_INCOME_SUBKIND_TO_CAT = {
    "salary": "salary",
    "interest": "investment",
    "dividend": "investment",
    "capital_gain": "investment",
    "ss_benefit": "social security",
}
_INCOME_CATS = ["salary", "social security", "investment"]

xfer = rc.items("transfer_event", from_kind="income").copy()
xfer["month"] = pd.to_datetime(xfer["month"])
xfer["category"] = xfer["from_subkind"].map(_INCOME_SUBKIND_TO_CAT).fillna("other")

income = (
    xfer.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_INCOME_CATS, fill_value=0)
    / 100
)
income["total"] = income.sum(axis=1)
income.head()


In [ ]:
# Only use current-run transfer-event income for the annual chart.
# Prior tax activity is pre-simulation YTD data that can distort the trend when
# folded into the same year totals.
income_yearly = income.groupby(income.index.year).sum()
income_yearly.index.name = "year"
income_yearly["total"] = income_yearly[_INCOME_CATS].sum(axis=1)
income_yearly.style.format("${:,.0f}")


In [ ]:
categories = ["salary", "social security", "investment"]
fig, ax = plt.subplots(figsize=(10, 5))
income_yearly[categories].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1f77b4", "#2ca02c", "#ff7f0e"],
)
ax.set_title(
    f"Annual income by category — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="category")
plt.show()


## Expenses by category

Expense flows are transfers _into_ accounts with `kind = 'expense'`. Subkinds are bucketed:

- **housing** — `housing`, `interest_expense`
- **healthcare** — `healthcare`
- **taxes** — `taxes_federal`, `taxes_state`, `taxes_property`
- **living** — `living`
- **other** — `other_expense`


In [ ]:
_EXPENSE_SUBKIND_TO_CAT = {
    "housing": "housing",
    "interest_expense": "housing",
    "taxes_property": "housing",
    "healthcare": "healthcare",
    "taxes_federal": "taxes",
    "taxes_state": "taxes",
    "living": "living",
    "food": "living",
}
_EXPENSE_CATS = ["housing", "healthcare", "taxes", "living", "other"]

xfer_exp = rc.items("transfer_event", to_kind="expense").copy()
xfer_exp["month"] = pd.to_datetime(xfer_exp["month"])
xfer_exp["category"] = (
    xfer_exp["to_subkind"].map(_EXPENSE_SUBKIND_TO_CAT).fillna("other")
)

expenses = (
    xfer_exp.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_EXPENSE_CATS, fill_value=0)
    / 100
)
expenses["total"] = expenses.sum(axis=1)
expenses_yearly = expenses.groupby(expenses.index.year).sum()
expenses_yearly.index.name = "year"
expenses_yearly.style.format("${:,.0f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
expenses_yearly[_EXPENSE_CATS].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1f77b4", "#d62728", "#9467bd", "#8c564b", "#7f7f7f"],
)
ax.set_title(
    f"Annual expenses by category — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="category")
plt.show()


## Net income

Net income = total income (incl. prior-year tax activity) − total expenses, by year.


In [ ]:
net = pd.DataFrame({
    "income": income_yearly["total"],
    "expenses": expenses_yearly["total"],
})
net["net"] = net["income"].fillna(0) - net["expenses"].fillna(0)
net.style.format("${:,.0f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
years = net.index.astype(str)
ax.bar(years, net["income"], label="income", color="#2ca02c", alpha=0.7)
ax.bar(years, -net["expenses"], label="expenses", color="#d62728", alpha=0.7)
ax.plot(years, net["net"], label="net", color="black", marker="o", linewidth=2)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title(
    f"Annual net income — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=90)
plt.show()
